In [10]:
import sys
import os

# Make config.py (in the repo root, one level up) importable
sys.path.append(os.path.abspath(".."))

import pandas as pd
import config

print("Config loaded.")
print("Clip catalog path:", config.CLIP_CATALOG_PATH)

Config loaded.
Clip catalog path: /Users/qian/KWF/rainforest-audio-detection/data/clip_catalog.csv


In [11]:
# Load the full clip catalog
df = pd.read_csv(config.CLIP_CATALOG_PATH)
print("Full catalog loaded:", df.shape)

# Columns we need for labeling
keep_cols = [
    "clip_name", "Recorder", "Timestamp", "Datetime", "Time Of Day",
    "Sim Type", "species", "confidence","audio_path",
]

# Trimmed working copy
labels = df[keep_cols].copy()
print("Trimmed working copy:", labels.shape)
labels.head()

Full catalog loaded: (631317, 33)
Trimmed working copy: (631317, 9)


,clip_name,Recorder,Timestamp,Datetime,Time Of Day,Sim Type,species,confidence,audio_path
0,Audio_Moth_1_20250317_093112.wav,Audio_Moth_1,20250317_093112,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
1,Audio_Moth_1_20250317_093115.wav,Audio_Moth_1,20250317_093115,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
2,Audio_Moth_1_20250317_093118.wav,Audio_Moth_1,20250317_093118,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
3,Audio_Moth_1_20250317_093121.wav,Audio_Moth_1,20250317_093121,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
4,Audio_Moth_1_20250317_093124.wav,Audio_Moth_1,20250317_093124,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...


In [12]:
# Initialize label columns — everything starts as "unknown"
labels["meaningful"] = "unknown"
labels["meaningful_source"] = "unlabeled"

# Confirm the starting state
print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown    631317
Name: count, dtype: int64

meaningful_source
unlabeled    631317
Name: count, dtype: int64


In [13]:
# Human activity carve-out: any clip in a sim event window is meaningful
human_mask = labels["Sim Type"] != "[]"

labels.loc[human_mask, "meaningful"] = "meaningful"
labels.loc[human_mask, "meaningful_source"] = "human_activity"

print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown       620446
meaningful     10871
Name: count, dtype: int64

meaningful_source
unlabeled         620446
human_activity     10871
Name: count, dtype: int64


In [14]:
import os

# Save intermediate labeling progress (this is an output, git-ignored)
out_dir = os.path.join(config.PROJECT_ROOT, "outputs")
os.makedirs(out_dir, exist_ok=True)

save_path = os.path.join(out_dir, "labels_progress.csv")
labels.to_csv(save_path, index=False)
print("Saved to:", save_path)

Saved to: /Users/qian/KWF/rainforest-audio-detection/outputs/labels_progress.csv


In [15]:
# Look at clips that have a named species
has_species = labels[labels["species"] != "[]"]
print("Clips with a named species:", len(has_species))
print()

# Show a few examples of species + confidence together
has_species[["clip_name", "species", "confidence", "audio_path"]].head(10)

Clips with a named species: 108855



,clip_name,species,confidence,audio_path
105,Audio_Moth_1_20250317_093627.wav,['Hylophylax naevioides_Spotted Antbird'],0.341533,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
114,Audio_Moth_1_20250317_093654.wav,['Mionectes olivaceus_Olive-striped Flycatcher'],0.267438,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
128,Audio_Moth_1_20250317_093736.wav,['Mionectes olivaceus_Olive-striped Flycatcher'],0.284766,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
208,Audio_Moth_1_20250317_094136.wav,['Malacoptila panamensis_White-whiskered Puffb...,0.463547,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
347,Audio_Moth_1_20250317_094833.wav,['Thamnophilus bridgesi_Black-hooded Antshrike'],0.560182,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
350,Audio_Moth_1_20250317_094842.wav,['Thamnophilus bridgesi_Black-hooded Antshrike'],0.287037,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
375,Audio_Moth_1_20250317_094957.wav,['Claravis pretiosa_Blue Ground Dove'],0.255881,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
376,Audio_Moth_1_20250317_095000.wav,['Myiothlypis fulvicauda_Buff-rumped Warbler'],0.948536,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
409,Audio_Moth_1_20250317_095139.wav,"[""Trogon bairdii_Baird's Trogon""]",0.259667,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
410,Audio_Moth_1_20250317_095142.wav,"[""Trogon bairdii_Baird's Trogon""]",0.351265,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...


In [18]:
# confidence is already a clean float — no parsing needed
labels["birdnet_conf"] = labels["confidence"]

# Look only at clips with a named species
species_clips = labels[labels["species"] != "[]"].copy()

# Bin the confidence the same way as the FP study
bins = [0.0, 0.3, 0.5, 0.7, 0.9, 1.01]
bin_labels = ["0.0-0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", "0.9-1.0"]
species_clips["conf_bin"] = pd.cut(species_clips["birdnet_conf"], bins=bins, labels=bin_labels, right=False)

print("Named-species clips by confidence bin:")
print(species_clips["conf_bin"].value_counts().sort_index())

Named-species clips by confidence bin:
conf_bin
0.0-0.3    23000
0.3-0.5    45176
0.5-0.7    20132
0.7-0.9    12090
0.9-1.0     8457
Name: count, dtype: int64


In [19]:
print("Total clips:", len(labels))
print("Clips with a valid audio_path:", labels["audio_path"].notna().sum())
print("Clips with missing audio_path:", labels["audio_path"].isna().sum())
print()
print("Example path:")
print(labels["audio_path"].iloc[0])

Total clips: 631317
Clips with a valid audio_path: 631317
Clips with missing audio_path: 0

Example path:
/Users/qian/KWF/Segmented_Foldered/Audio Moth 1/0_4999/Audio_Moth_1_20250317_093112.wav


In [20]:
# BirdNET named-species audit sampler
import os
import shutil
import numpy as np

# --- Config ---
N_PER_BIN = 20
RANDOM_SEED = 42

OUT_DIR = os.path.join(config.PROJECT_ROOT, "outputs", "birdnet_audit")
AUDIO_OUT = os.path.join(OUT_DIR, "audio")
os.makedirs(AUDIO_OUT, exist_ok=True)

# --- Sample N_PER_BIN from each confidence bin ---
rng = np.random.RandomState(RANDOM_SEED)
sampled = []
for b in bin_labels:
    cell = species_clips[species_clips["conf_bin"] == b]
    n_take = min(N_PER_BIN, len(cell))
    sampled.append(cell.sample(n=n_take, random_state=rng))

sample_df = pd.concat(sampled, ignore_index=True)
print("Total sampled:", len(sample_df))
print(sample_df["conf_bin"].value_counts().sort_index())

# --- Copy audio files, build review rows ---
review_rows = []
missing = 0
for _, row in sample_df.iterrows():
    src = row["audio_path"]
    clip_name = row["clip_name"]
    out_name = "{}_{}".format(row["conf_bin"], clip_name)

    if os.path.exists(src):
        shutil.copy2(src, os.path.join(AUDIO_OUT, out_name))
    else:
        missing += 1
        continue

    review_rows.append({
        "clip_name": clip_name,
        "recorder": row["Recorder"],
        "conf_bin": row["conf_bin"],
        "birdnet_conf": row["birdnet_conf"],
        "species": row["species"],
        "audio_file": "audio/" + out_name,
        "category": "",   # fill while listening: bird / no-bird / unsure
        "notes": "",
    })

review_df = pd.DataFrame(review_rows)
review_df = review_df.sort_values(["conf_bin", "birdnet_conf"], ascending=[True, False])

csv_path = os.path.join(OUT_DIR, "birdnet_audit.csv")
review_df.to_csv(csv_path, index=False)

print("\nDone.")
print("  Audio copied to:", AUDIO_OUT)
print("  Review CSV:     ", csv_path)
if missing:
    print(f"  [WARN] {missing} audio files missing from source")

Total sampled: 100
conf_bin
0.0-0.3    20
0.3-0.5    20
0.5-0.7    20
0.7-0.9    20
0.9-1.0    20
Name: count, dtype: int64

Done.
  Audio copied to: /Users/qian/KWF/rainforest-audio-detection/outputs/birdnet_audit/audio
  Review CSV:      /Users/qian/KWF/rainforest-audio-detection/outputs/birdnet_audit/birdnet_audit.csv


In [ ]:
import os
import pandas as pd
from IPython.display import display, Audio, clear_output
import ipywidgets as widgets

# Paths
AUDIT_DIR = os.path.join(config.PROJECT_ROOT, "outputs", "birdnet_audit")
AUDIT_CSV = os.path.join(AUDIT_DIR, "birdnet_audit.csv")

# Load
adf = pd.read_csv(AUDIT_CSV)
adf["category"] = adf["category"].fillna("").astype(str)
adf["notes"] = adf["notes"].fillna("").astype(str)

print(f"Total clips to review: {len(adf)}")
print(f"Already tagged: {(adf['category'] != '').sum()}")
print(f"Remaining: {(adf['category'] == '').sum()}")

current_idx = [0]
CATEGORIES = ["bird", "no-bird", "unsure"]

category_dropdown = widgets.Dropdown(options=[""] + CATEGORIES, description="Category:", value="")
notes_text = widgets.Text(description="Notes:", placeholder="optional")
prev_btn = widgets.Button(description="◀ Prev")
next_btn = widgets.Button(description="Next ▶", button_style="primary")
save_btn = widgets.Button(description="💾 Save CSV", button_style="success")
jump_input = widgets.IntText(value=0, description="Jump to:")
jump_btn = widgets.Button(description="Go")
status_label = widgets.Label(value="")
output = widgets.Output()

def show_clip(idx):
    with output:
        clear_output(wait=True)
        if idx < 0 or idx >= len(adf):
            print("Out of range.")
            return
        row = adf.iloc[idx]
        n_tagged = (adf["category"] != "").sum()
        print(f"Clip {idx + 1} / {len(adf)}  |  Tagged so far: {n_tagged} / {len(adf)}")
        print(f"File: {row['clip_name']}")
        print(f"Recorder: {row['recorder']}  |  Conf bin: {row['conf_bin']}  |  BirdNET conf: {row['birdnet_conf']:.3f}")
        print(f"BirdNET claimed species: {row['species']}")
        print(f"Current tag: '{row['category']}'  |  Notes: '{row['notes']}'")
        print()
        audio_path = os.path.join(AUDIT_DIR, row["audio_file"])
        if os.path.exists(audio_path):
            display(Audio(filename=audio_path))
        else:
            print(f"[Audio not found: {audio_path}]")
    category_dropdown.value = row["category"] if row["category"] in [""] + CATEGORIES else ""
    notes_text.value = row["notes"]

def save_current_tags():
    adf.at[current_idx[0], "category"] = category_dropdown.value
    adf.at[current_idx[0], "notes"] = notes_text.value

def on_next(b):
    save_current_tags()
    if current_idx[0] < len(adf) - 1:
        current_idx[0] += 1
        show_clip(current_idx[0])
    else:
        status_label.value = "Reached the end. Don't forget to save."

def on_prev(b):
    save_current_tags()
    if current_idx[0] > 0:
        current_idx[0] -= 1
        show_clip(current_idx[0])

def on_save(b):
    save_current_tags()
    adf.to_csv(AUDIT_CSV, index=False)
    n_tagged = (adf["category"] != "").sum()
    status_label.value = f"✓ Saved. {n_tagged}/{len(adf)} tagged."

def on_jump(b):
    save_current_tags()
    target = jump_input.value
    if 0 <= target < len(adf):
        current_idx[0] = target
        show_clip(current_idx[0])

next_btn.on_click(on_next)
prev_btn.on_click(on_prev)
save_btn.on_click(on_save)
jump_btn.on_click(on_jump)

controls = widgets.HBox([prev_btn, next_btn, save_btn])
jump_row = widgets.HBox([jump_input, jump_btn])
ui = widgets.VBox([output, category_dropdown, notes_text, controls, jump_row, status_label])

display(ui)
show_clip(current_idx[0])

Total clips to review: 100
Already tagged: 0
Remaining: 100


In [22]:
# BirdNET named-species carve-out: any clip with a named species is meaningful.
# Only label clips still "unknown" — human_activity (gold standard) keeps its source.
species_mask = (labels["species"] != "[]") & (labels["meaningful"] == "unknown")

labels.loc[species_mask, "meaningful"] = "meaningful"
labels.loc[species_mask, "meaningful_source"] = "birdnet_species"

print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown       512377
meaningful    118940
Name: count, dtype: int64

meaningful_source
unlabeled          512377
birdnet_species    108069
human_activity      10871
Name: count, dtype: int64


In [23]:
# Of the unknown pool, how many have BirdNET confidence exactly 0?
unknown_pool = labels[labels["meaningful"] == "unknown"]
print("Unknown pool size:", len(unknown_pool))

zero_conf = unknown_pool[unknown_pool["birdnet_conf"] == 0.0]
print("Unknown clips with BirdNET confidence = 0:", len(zero_conf))

# Also show the confidence distribution in the unknown pool, to see the shape
print()
print("Confidence distribution in unknown pool:")
print(unknown_pool["birdnet_conf"].describe())

Unknown pool size: 512377
Unknown clips with BirdNET confidence = 0: 512377

Confidence distribution in unknown pool:
count    512377.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: birdnet_conf, dtype: float64
